<a href="https://colab.research.google.com/github/sarahibdah/Flyrank-ML-Internship-Sarah/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sarahibdah/Flyrank-ML-Internship-Sarah/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/sarahibdah/Flyrank-ML-Internship-Sarah/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print(f"Loaded {len(df):,} pages")

Loaded 30,000 pages


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method: Decision Tree Classifier.

I'm using a Decision Tree because it directly extends my Week-4 baseline rule, which was itself a hand-written if/then tree: risky age AND declining → refresh_now. A trained Decision Tree lets the data discover its own splits and thresholds automatically, rather than me hand-picking them, while staying interpretable — I can inspect exactly which features it split on and why, which matters for editor trust in a real refresh-calendar tool.

This fits my task type (classification/scoring, established in ML-03) and my data grain (one row = one page). I'll keep the tree shallow (max_depth 3-5) to avoid overfitting — the same discipline I practiced in my ML-04 leakage exercise, where a shallow tree made leakage harder to exploit and kept the model honest.

I'm avoiding Random Forest/Gradient Boosting for this first model — they're more powerful, but harder to interpret and easier to over-trust without first establishing a clean, explainable baseline model. Since the model must beat my Week-4 rule specifically, starting with something structurally similar (a tree) makes the comparison meaningful, not just "complex model vs simple rule" but "learned tree vs hand-written tree."

In [ ]:
# Confirming the data is shaped correctly for a Decision Tree classifier
print("Features available for modeling:")
print(df[["age_tier_order", "impressions_90d", "trend_direction"]].dtypes)

print(f"\nClass balance (declining vs not):")
print(df["trend_direction"].value_counts(normalize=True))

print(f"\nTotal pages available: {len(df):,}")

Features available for modeling:
age_tier_order      int64
impressions_90d     int64
trend_direction    object
dtype: object

Class balance (declining vs not):
trend_direction
down      0.542067
stable    0.198733
up        0.146267
new       0.074533
flat      0.038400
Name: proportion, dtype: float64

Total pages available: 30,000


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split design: grouped by client.

Since pages from the same client share context (site quality, industry, content style — as I observed in my Week-4 baseline's client-clustering pattern, where a handful of clients dominated my top-20 picks), a random split could let the model learn client-specific quirks and get evaluated on the same client, inflating its apparent performance.

I'm splitting by client_id: roughly 80% of clients (26 of 32) go to training, the remaining ~6 clients go to testing entirely. This ensures the model is evaluated on clients it has never seen before, which is a stronger, more honest test of generalization — closer to how the model would actually be used on a brand-new client in production. This follows the same discipline as my ML-04 leakage lesson: no information from the test set should be available to the model during training, whether that's a future time window or, in this case, a client's own patterns.

In [ ]:
import numpy as np

np.random.seed(42)
all_clients = df["client_id"].unique()
np.random.shuffle(all_clients)

split_idx = int(len(all_clients) * 0.8)
train_clients = all_clients[:split_idx]
test_clients = all_clients[split_idx:]

train_df = df[df["client_id"].isin(train_clients)].copy()
test_df = df[df["client_id"].isin(test_clients)].copy()

print(f"Train: {len(train_df):,} pages from {len(train_clients)} clients")
print(f"Test: {len(test_df):,} pages from {len(test_clients)} clients")

Train: 22,389 pages from 25 clients
Test: 7,611 pages from 7 clients


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I trained a Decision Tree (max_depth=4) using two features — age_tier_order and impressions_90d — on the 25 training clients from my Section 2 split. The target is whether a page is declining (trend_direction == "down"), matching my ML-03 framing.

To make the comparison fair, I evaluated both the trained tree and my exact Week-4 baseline rule (risky age tier AND declining → flagged) on the same held-out test set — the 7 clients the model never saw during training. Both were scored using precision: of the pages each method flagged, how many were genuinely declining.

Results: Decision Tree precision = [X], Week-4 baseline precision = [Y].

[If the tree wins:] The trained tree slightly outperforms my hand-written rule, likely because it can find a more precise age/impressions threshold than my manually chosen tier cutoffs.

[If the baseline wins or ties:] My hand-written rule performs comparably to (or better than) the trained tree. This isn't a failure — it suggests the signal in these two features is fairly simple and linear, and a hand-picked threshold captures it about as well as a learned one. Per the card's own warning, this doesn't mean I should reach for a more complex model just to "win" — it means my baseline was already a reasonably good encoding of the real pattern.




In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score
import pandas as pd

feature_cols = ["age_tier_order", "impressions_90d"]
X_train = train_df[feature_cols]
y_train = (train_df["trend_direction"] == "down").astype(int)
X_test = test_df[feature_cols]
y_test = (test_df["trend_direction"] == "down").astype(int)

tree = DecisionTreeClassifier(max_depth=4, random_state=42)
tree.fit(X_train, y_train)
tree_preds = tree.predict(X_test)
tree_precision = precision_score(y_test, tree_preds)

# FIXED baseline: predicts decline using ONLY age (no peeking at trend_direction)
test_df = test_df.copy()
test_df["baseline_flagged"] = test_df["age_tier_order"].isin([3, 4]).astype(int)
baseline_precision = precision_score(y_test, test_df["baseline_flagged"])

comparison = pd.DataFrame({
    "Method": ["Week-4 Baseline (age only, fair)", "Decision Tree (max_depth=4)"],
    "Precision": [baseline_precision, tree_precision],
    "Pages flagged": [test_df["baseline_flagged"].sum(), tree_preds.sum()],
    "Test set size": [len(test_df), len(test_df)],
})

print("Model vs. Baseline — same test set, same metric, FAIR comparison:")
print(comparison.to_string(index=False))

Model vs. Baseline — same test set, same metric, FAIR comparison:
                          Method  Precision  Pages flagged  Test set size
Week-4 Baseline (age only, fair)   0.627232           3584           7611
     Decision Tree (max_depth=4)   0.622940           5946           7611


In [ ]:
from sklearn.metrics import recall_score

tree_recall = recall_score(y_test, tree_preds)
baseline_recall = recall_score(y_test, test_df["baseline_flagged"])

print(f"Decision Tree recall: {tree_recall:.3f}")
print(f"Baseline recall:      {baseline_recall:.3f}")

Decision Tree recall: 0.916
Baseline recall:      0.556


Fair comparison result: with the baseline corrected to use only age_tier_order (removing the earlier circular label-peeking), precision is nearly tied — Decision Tree 0.623 vs. Baseline 0.627. But recall tells a different story: the Decision Tree recalls 91.6% of actual declining pages, versus the baseline's 55.6% — the baseline misses nearly half of all real decliners, while the tree misses fewer than 1 in 10.

This is a genuine, meaningful improvement, and it comes from a specific, explainable source: the tree combines two validated signals (age and impressions volume) rather than relying on age alone. This matches my Week-4 Signal 2 finding — volume was MIXED as a standalone risk predictor, but clearly adds real value when combined with age. The model beats the baseline not through raw complexity, but by using more of the evidence I'd already gathered and validated.

In [ ]:
from sklearn.tree import export_text
tree_rules = export_text(tree, feature_names=feature_cols)
print(tree_rules)

|--- impressions_90d <= 5.50
|   |--- impressions_90d <= 3.50
|   |   |--- age_tier_order <= 3.50
|   |   |   |--- impressions_90d <= 1.50
|   |   |   |   |--- class: 0
|   |   |   |--- impressions_90d >  1.50
|   |   |   |   |--- class: 0
|   |   |--- age_tier_order >  3.50
|   |   |   |--- age_tier_order <= 5.50
|   |   |   |   |--- class: 0
|   |   |   |--- age_tier_order >  5.50
|   |   |   |   |--- class: 0
|   |--- impressions_90d >  3.50
|   |   |--- age_tier_order <= 3.50
|   |   |   |--- impressions_90d <= 4.50
|   |   |   |   |--- class: 1
|   |   |   |--- impressions_90d >  4.50
|   |   |   |   |--- class: 0
|   |   |--- age_tier_order >  3.50
|   |   |   |--- age_tier_order <= 5.50
|   |   |   |   |--- class: 0
|   |   |   |--- age_tier_order >  5.50
|   |   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- age_tier_order <= 5.50
|   |   |--- age_tier_order <= 4.50
|   |   |   |--- impressions_90d <= 70.50
|   |   |   |   |--- class: 1
|   |   |   |--- impressions

 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

What does the model lean on?

Reading the tree's actual decision logic (export_text), it relies much more heavily on impressions_90d than age_tier_order — impressions appears as the deciding check in nearly every branch, often multiple times in a row.

A significant portion of that reliance looks like overfitting rather than real signal: many splits distinguish between pages with just 1, 2, 3, or 4 impressions — differences too small to represent a meaningful pattern about whether a page is genuinely declining. A page's traffic barely registers at that scale, so the tree is likely memorizing noise specific to my training data rather than learning something that will generalize.

However, the tree also contains more trustworthy, meaningful splits — for example, pages older than 365 days with impressions roughly between 1,000-4,800 get flagged as declining. This combines both signals in a way my simple age-only baseline couldn't express, and matches my Week-4 finding that age and volume interact rather than acting independently.

This suggests a shallower or more constrained tree (e.g. min_samples_leaf set higher, forcing each split to represent a meaningful number of real pages) would likely generalize better by avoiding the noisy micro-splits on near-zero-traffic pages, while keeping the genuinely useful age/impressions interaction.

In [ ]:
test_df["predicted"] = tree_preds
test_df["actual"] = y_test.values

false_negatives = test_df[(test_df["predicted"] == 0) & (test_df["actual"] == 1)]
false_positives = test_df[(test_df["predicted"] == 1) & (test_df["actual"] == 0)]

print(f"False negatives (missed real decliners): {len(false_negatives):,}")
print(f"False positives (wrongly flagged): {len(false_positives):,}")

print(f"\nFalse negative examples (model missed these):")
print(false_negatives[["age_tier_order", "impressions_90d"]].describe())

print(f"\nFalse positive examples (model wrongly flagged these):")
print(false_positives[["age_tier_order", "impressions_90d"]].describe())

False negatives (missed real decliners): 341
False positives (wrongly flagged): 2,242

False negative examples (model missed these):
       age_tier_order  impressions_90d
count      341.000000       341.000000
mean         5.398827       141.577713
std          0.588516       761.863658
min          4.000000         1.000000
25%          5.000000         5.000000
50%          5.000000        16.000000
75%          6.000000        60.000000
max          6.000000     10186.000000

False positive examples (model wrongly flagged these):
       age_tier_order  impressions_90d
count     2242.000000      2242.000000
mean         4.453167     10125.044603
std          0.559524     25239.831748
min          3.000000         6.000000
25%          4.000000       825.750000
50%          4.000000      2977.500000
75%          5.000000      9748.500000
max          6.000000    497727.000000


Where is the model wrong?

The model has 341 false negatives (missed real decliners) and 2,242 false positives (wrongly flagged stable pages) — false positives are the dominant error type, roughly 6x more common.

False negatives cluster around older pages (age tier 5-6, i.e. 181+ days) with very low traffic — median only 16 impressions. This matches the noise issue from Part 1: pages with barely any traffic don't give the tree enough signal to detect a real decline, so it tends to default to "not declining" even when they are.

False positives cluster around younger-to-mid-age pages (tier 4-5) with substantial traffic — median 2,977 impressions, some as high as nearly 500,000. This suggests the tree over-generalizes the pattern from my Week-4 Signal 2 finding (mid-traffic tiers had the highest decline rates, 60-63%) — it's flagging many large, established pages as declining when a meaningful share of them are actually stable.
In short: the model trades a smaller number of missed low-traffic decliners for a much larger number of false alarms on bigger, mid-age pages. For a real refresh calendar, this means editors would need to sanity-check flagged high-traffic pages before committing resources, since roughly 6x more get wrongly flagged than genuinely missed.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.